# Learning Faster R-CNN with PennFudanPed



## 1. Setup

In [ ]:
import os, zipfile, urllib.request
import torch
import matplotlib.pyplot as plt
from torchvision.io import read_image
from torchvision.ops import masks_to_boxes
from torchvision import tv_tensors
from torchvision.transforms import v2 as T
from torchvision.transforms.v2 import functional as F
from torchvision.utils import draw_bounding_boxes , save_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

### Importing Data

In [ ]:
import os

if os.path.isdir("PennFudanPed/PNGImages"):      # notebook is next to the PennFudanPed folder
    ROOT = "PennFudanPed"
elif os.path.isdir("PNGImages"):                 # notebook is inside the PennFudanPed folder
    ROOT = "."
else:
    raise FileNotFoundError(f"PNGImages not found. Current folder: {os.getcwd()}")

print("ROOT =", ROOT)
print(len(os.listdir(os.path.join(ROOT, "PNGImages"))), "images,",
      len(os.listdir(os.path.join(ROOT, "PedMasks"))), "masks")

## 3. Dataset class

only need boxes (not masks), so we derive them from the mask file.

In [ ]:
class PennFudanDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms
        self.imgs = sorted(os.listdir(os.path.join(root, "PNGImages")))
        self.masks = sorted(os.listdir(os.path.join(root, "PedMasks")))

    def __getitem__(self, idx):
        img = read_image(os.path.join(self.root, "PNGImages", self.imgs[idx]))   # uint8 [3,H,W]
        mask = read_image(os.path.join(self.root, "PedMasks", self.masks[idx]))  # [1,H,W]

        obj_ids = torch.unique(mask)[1:]                       # drop 0 = background
        masks = (mask == obj_ids[:, None, None]).to(torch.uint8)  # [N,H,W] one mask per person
        boxes = masks_to_boxes(masks)                          # [N,4] xyxy
        labels = torch.ones((len(obj_ids),), dtype=torch.int64)

        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=F.get_size(img)),
            "labels": labels,
        }
        if self.transforms is not None:
            img, target = self.transforms(img, target)
        return img, target

    def __len__(self):
        return len(self.imgs)

### Look at a sample (always visualize your data first)

In [ ]:
ds = PennFudanDataset(ROOT)
img, target = ds[10]
vis = draw_bounding_boxes(img, target["boxes"], colors="red", width=3)
plt.figure(figsize=(6, 8)); plt.imshow(vis.permute(1, 2, 0)); plt.axis("off")
print("num people:", len(target["boxes"]), "| image size:", img.shape)

## 4. Transforms, split, DataLoaders


In [ ]:
def get_transform(train):
    t = []
    if train:
        t.append(T.RandomHorizontalFlip(0.5))
    t.append(T.ToDtype(torch.float, scale=True))
    t.append(T.ToPureTensor())
    return T.Compose(t)

def collate_fn(batch):
    # images have different sizes -> keep them as tuples, not stacked tensors
    return tuple(zip(*batch))

train_ds = PennFudanDataset(ROOT, get_transform(train=True))
test_ds  = PennFudanDataset(ROOT, get_transform(train=False))

g = torch.Generator().manual_seed(42)
indices = torch.randperm(len(train_ds), generator=g).tolist()
train_ds = torch.utils.data.Subset(train_ds, indices[:-50])
test_ds  = torch.utils.data.Subset(test_ds,  indices[-50:])

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=2, shuffle=True,  num_workers=0, collate_fn=collate_fn)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)
print(len(train_ds), "train /", len(test_ds), "test")

## 5. Model: pretrained Faster R-CNN with a new head


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_classes=2):
    model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = get_model().to(device)
print(model.roi_heads.box_predictor)

## 6. Training


In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

num_epochs = 15  

for epoch in range(num_epochs):
    model.train()
    running = {}
    for images, targets in train_loader:
        images = [im.to(device) for im in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        for k, v in loss_dict.items():
            running[k] = running.get(k, 0) + v.item()
    lr_scheduler.step()

    n = len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | total {sum(running.values())/n:.3f} | " +
          " | ".join(f"{k.replace('loss_', '')} {v/n:.3f}" for k, v in running.items()))

## 7. Inference & visualization


In [ ]:
@torch.no_grad()
def show_prediction(dataset, idx, threshold=0.7):
    model.eval()
    img, _ = dataset[idx]
    pred = model([img.to(device)])[0]
    keep = pred["scores"] > threshold
    boxes = pred["boxes"][keep].cpu()
    labels = [f"person {s:.2f}" for s in pred["scores"][keep].cpu()]
    img_u8 = (img * 255).to(torch.uint8)
    vis = draw_bounding_boxes(img_u8, boxes, labels=labels, colors="lime", width=1)
    plt.figure(figsize=(6, 8)); plt.imshow(vis.permute(1, 2, 0)); plt.axis("off")
    # plt.imshow(vis.permute(1, 2, 0)); plt.axis("off")

    save_path = os.path.join(output_dir, f"prediction_{idx}.png")
    save_image(vis.float() / 255, save_path)

for i in range(15):
    show_prediction(test_ds, i)


## 8. Evaluation: mean Average Precision (mAP)



In [ ]:
from torchmetrics.detection import MeanAveragePrecision

metric = MeanAveragePrecision(iou_type="bbox")
model.eval()
with torch.no_grad():
    for images, targets in test_loader:
        images = [im.to(device) for im in images]
        preds = model(images)
        preds = [{k: v.cpu() for k, v in p.items()} for p in preds]
        gts = [{"boxes": t["boxes"].as_subclass(torch.Tensor), "labels": t["labels"]} for t in targets]
        metric.update(preds, gts)

res = metric.compute()
print("mAP:", res["map"].item(), "| mAP@0.5:", res["map_50"].item())

## 9. Saving the model

In [ ]:
torch.save(model.state_dict(), "fasterrcnn_pennfudan.pth")